<a href="https://colab.research.google.com/github/AIVIETNAM-AIO-Tuan/AIO-Conquer/blob/premain/Model/Riboflavin.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
!pip install Boruta -q

import pandas as pd
import numpy as np
from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from boruta import BorutaPy
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LassoCV

In [21]:
drive.mount('/content/drive')
# FILE_PATH = '/content/drive/MyDrive/Colab Notebooks/Data/riboflavin_processed.csv'
FILE_PATH = '/content/drive/MyDrive/AIO-Conquer02/riboflavin_processed.csv'
try:
    df = pd.read_csv(FILE_PATH)
    print(f"Đã tải dữ liệu thành công! Kích thước: {df.shape}")
    df.info()
    df = df.copy()
except Exception as e:
    print(f"Lỗi: Không tìm thấy file hoặc đường dẫn sai. Chi tiết: {e}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Đã tải dữ liệu thành công! Kích thước: (71, 4089)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Columns: 4089 entries, y to x.zur_at
dtypes: float64(4089)
memory usage: 2.2 MB


In [22]:
target_colum = 'y'
X = df.drop(columns=[target_colum], errors='ignore')
y = df[target_colum]

#Chia tập Train/Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [23]:
# 1. Khởi tạo mô hình
model = LinearRegression()

# 2. Huấn luyện trên toàn bộ đặc trưng
model.fit(X_train, y_train)

# 3. Dự đoán và đánh giá
y_pred = model.predict(X_test)
r2   = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)
#4. In kết quả
print(f"R²   : {r2:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")

R²   : 0.6498
RMSE : 0.4502
MAE  : 0.3391


In [24]:
# 1. Khởi tạo bộ ước lượng nền và bộ chọn đặc trưng Boruta
rf_estimator = RandomForestRegressor(n_jobs=-1, max_depth=5, random_state=42)
selector = BorutaPy(
    estimator=rf_estimator,
    n_estimators='auto',
    max_iter=100,
    random_state=42
)

# 2. Huấn luyện Boruta trên tập train (yêu cầu mảng NumPy)
selector.fit(X_train.values, y_train.values)

# 3. Lọc đặc trưng được xác nhận (confirmed features)
X_train_selected = selector.transform(X_train.values)
X_test_selected  = selector.transform(X_test.values)
selected_features = X_train.columns[selector.support_].tolist()

# 4. Huấn luyện Linear Regression trên tập đặc trưng đã chọn
model = LinearRegression()
model.fit(X_train_selected, y_train.values)

# 5. Dự đoán và đánh giá
y_pred = model.predict(X_test_selected)
r2   = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)

#6. In kết quả
print(f"R²   : {r2:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")

feature_names = X_train.columns
confirmed_features = feature_names[selector.support_].tolist()

print(f"So luong dac trung duoc Boruta giu lai: {len(confirmed_features)}")
print(f"Danh sach dac trung: {confirmed_features}")

R²   : 0.4718
RMSE : 0.5529
MAE  : 0.4632
So luong dac trung duoc Boruta giu lai: 13
Danh sach dac trung: ['x.PHRI_r_at', 'x.PKSA_at', 'x.PROJ_at', 'x.XLYB_at', 'x.YCKE_at', 'x.YDDK_at', 'x.YHCL_at', 'x.YHFH_r_at', 'x.YHFU_at', 'x.YLAJ_at', 'x.YNZA_at', 'x.YXLD_at', 'x.YXLE_at']


In [25]:
# 1. Chuẩn hóa đặc trưng (bắt buộc với Lasso)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# 2. Tìm alpha tối ưu bằng LassoCV (5-fold trên tập train)
lasso_cv = LassoCV(alphas=np.logspace(-2, -1, 100), cv=5, random_state=42, n_jobs=-1)
lasso_cv.fit(X_train_scaled, y_train)

# 3. Trích xuất đặc trưng được giữ lại (hệ số khác 0)
mask = lasso_cv.coef_ != 0
selected_features = X_train.columns[mask].tolist()

X_train_selected = X_train_scaled[:, mask]
X_test_selected  = X_test_scaled[:, mask]

# 4. Huấn luyện Linear Regression trên tập đặc trưng đã chọn
model = LinearRegression()
model.fit(X_train_selected, y_train)

# 5. Dự đoán và đánh giá
y_pred = model.predict(X_test_selected)
r2   = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)

# 6. In kết quả
n_selected = mask.sum()
n_total    = X_train.shape[1]

print(f"R²   : {r2:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"Số đặc trưng được giữ lại : {n_selected}/{n_total}")
print(f"Danh sách đặc trưng       : {selected_features}")

R²   : 0.6024
RMSE : 0.4797
MAE  : 0.4156
Số đặc trưng được giữ lại : 52/4088
Danh sách đặc trưng       : ['x.ACOA_at', 'x.ARAA_at', 'x.ARGB_at', 'x.ARGF_at', 'x.bhlA_i_at', 'x.GBSA_at', 'x.LYSC_at', 'x.PRIA_at', 'x.PTA_at', 'x.SACC_at', 'x.SPOIVA_at', 'x.SPOVAA_at', 'x.XKDB_at', 'x.XKDF_at', 'x.YACN_at', 'x.YBFI_at', 'x.YCLB_at', 'x.YCLF_at', 'x.YDDH_at', 'x.YDDK_at', 'x.YEBC_at', 'x.YETH_at', 'x.YFHE_r_at', 'x.YFII_at', 'x.YFIT_at', 'x.YFJD_at', 'x.YGAK_at', 'x.YHDS_r_at', 'x.YHDZ_at', 'x.YHZA_at', 'x.YIST_at', 'x.YISU_at', 'x.YKVJ_at', 'x.YLXW_at', 'x.YOAB_at', 'x.YOPS_at', 'x.YOSU_at', 'x.YOTH_i_at', 'x.YPGA_at', 'x.YPUD_at', 'x.YQJT_at', 'x.YQJU_at', 'x.YRZE_at', 'x.YTAB_at', 'x.YUID_at', 'x.YULC_at', 'x.YVOA_at', 'x.YWCI_at', 'x.YXAF_at', 'x.YXIE_at', 'x.YXLD_at', 'x.YYDA_at']
